# Study 1: User survey scores（研究1：用户调查评分）

Therapeutic alliance (WAI-SR), pre/post mood, and ratings of perceived agent "humanness".（治疗联盟（WAI-SR）韦式成人智力表、前后情绪变化，以及对智能体"人性化程度"的评分。）

In [7]:
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

import scipy.stats as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf

import plotly.express as px
import plotly.graph_objects as go

from utils.utils import *
from utils.variables import *

C:\Users\shishuaicheng\AppData\Local\Temp\ipykernel_26412\1413199018.py:3: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


## Read in data（读取数据）

Only analyze transcripts that expert consortium rated as being appropriate for CBT.（仅分析专家团队评定为适合CBT的对话记录。）

In [8]:
# 过滤掉不适合CBT的对话记录
df_pids, df_users = retrieve_analysed_data('users')

Removing 7 transcripts that are not appropriate for CBT
N = 227


condition        model 
cognitive_layer  claude    25
                 gemini    26
                 gpt4      26
                 llama     24
human_therapist  human     26
standalone_llms  claude    25
                 gemini    24
                 gpt4      27
                 llama     24
dtype: int64

## Therapeutic alliance（治疗联盟）

In [9]:
df = df_users.copy().merge(df_pids[['pid','condition','model']],on='pid',how='left')

cols = [x for x in df.columns if 'waisr_' in x]
for col in cols:
    df[col] = df[col].str.lower().replace({
        'strong disagree': 1,
        'disagree': 2,
        'neutral': 3,
        'agree': 4,
        'strong agree': 5,   
    }).astype(int)

df['waisr_overall'] = df[cols].mean(axis=1)

outcomes = ['overall','goal','task','bond']
for outcome in outcomes:
    df[f'waisr_{outcome}'] = df[[x for x in df.columns if f'waisr_{outcome}' in x]].mean(axis=1)

# 打印平均值
display(df.groupby('condition')['waisr_overall'].agg(['mean','sem']).round(1))

# ------------------------------------------------------------------------------------------------------------------------------------
# 运行模型（按各子量表）
# ------------------------------------------------------------------------------------------------------------------------------------

min_pval_moderators = 1

for outcome in outcomes:

    print_title(f'WAI-SR - {outcome.upper()}')

    # --- 2x4 模型
    model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

    model_fit = smf.ols(f'waisr_{outcome} ~ C(model) * C(condition) * belief_human + C(model) * C(condition) * feel_human', data=model_df).fit()

    display_residuals(model_fit)
    anova_table = format_anova_table(model_fit)

    print('[2 x 4 model]')
    print(anova_table)

    if outcome=='overall':
        for contrast in ['C(condition):feel_human','C(condition):belief_human']:
            pval = anova_table.loc[contrast,'PR(>F)']
            min_pval_moderators = min(min_pval_moderators,pval)

    # --- 三因素模型
    model_df = df.groupby(['pid','condition'])[[f'waisr_{outcome}','belief_human','feel_human']].mean().reset_index()

    model_fit = smf.ols(f'waisr_{outcome} ~ C(condition)*belief_human + C(condition)*feel_human', data=model_df).fit()

    display_residuals(model_fit)
    anova_table = format_anova_table(model_fit)

    print('[3-way model]')
    print(anova_table)

    print(' ')
    print(anova_pairwise_comparisons(model_fit))

    if outcome=='overall':
        for contrast in ['C(condition):feel_human','C(condition):belief_human']:
            pval = anova_table.loc[contrast,'PR(>F)']
            min_pval_moderators = min(min_pval_moderators,pval)

        print(f"\nMin p-value for human perception moderators: {min_pval_moderators}")

# ------------------------------------------------------------------------------------------------------------------------------------
# 绘图
# ------------------------------------------------------------------------------------------------------------------------------------
# 跨模型平均
melted = (
    df
    .copy()
    .melt(id_vars=['pid','condition','model'],value_vars=[f'waisr_{x}' for x in outcomes],var_name='subscale',value_name='score')
)
fig = plot_continuous_dot_and_bar(
    melted,outcome_var='score',x_var='subscale',color_var='condition',
    y_range=(0,5.1),marker_opacity=0.2,
    color_order=CONDITION_ORDER,colour_map=COLOURS
    )

# 按LLM拆分（仅总量表）
fig = plot_continuous_dot_and_bar(
    melted.loc[melted['condition']!='human_therapist',],outcome_var='score',x_var='subscale',color_var='condition',
    y_range=(0,5.1),marker_opacity=0.2,
    color_order=CONDITION_ORDER,colour_map=COLOURS,
    facet_col='model', facet_col_wrap=2,
    width=600,height=500
    )
fig.update_yaxes(tickvals=[0,1,2,3,4,5])
fig.show()

# summary = (
#     df
#     .loc[df['condition']!='human_therapist',]
#     .copy()
#     .melt(id_vars=['pid','condition','model'],value_vars=[f'waisr_{x}' for x in outcomes],var_name='subscale',value_name='score')
#     .groupby(['condition','model','subscale'])['score'].agg(['mean','sem'])
#     .reset_index()
# )
# summary['subscale'] = summary['subscale'].str.replace('waisr_','').str.capitalize()

# fig = px.bar(
#     summary,
#     x='subscale',
#     y='mean',
#     facet_col='model',
#     facet_col_wrap=2,
#     error_y='sem',
#     color='condition',
#     title='WAI-SR',
#     barmode='group',
#     width=600,
#     height=500,
#     labels={'mean': 'Score'},
#     category_orders={
#         'condition': CONDITION_ORDER,
#         'subscale': ['Overall','Goal','Task','Bond']
#         },
#     color_discrete_map=COLOURS,
#     template='simple_white'
# )
# fig.update_layout(font={'family': 'Arial'})
# fig.update_yaxes(range=[1,5])
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.show()

# fig.write_image(f'../results/waisr_perllm.svg')


,mean,sem
condition,,
cognitive_layer,4.0,0.1
human_therapist,3.9,0.2
standalone_llms,3.7,0.1


WAI-SR - OVERALL


Shapiro-Wilk: 0.950, p = 1.93e-06***
Kolmogorov-Smirnov: 0.087, p = 0.09
[2 x 4 model]
                                       sum_sq     df          F        PR(>F)  \
C(model)                             2.337668    3.0   2.278159  8.121106e-02   
C(condition)                         2.358104    1.0   6.894223  9.404628e-03   
C(model):C(condition)                3.079643    3.0   3.001245  3.197213e-02   
belief_human                         0.692493    1.0   2.024593  1.565305e-01   
C(model):belief_human                0.864355    3.0   0.842351  4.723620e-01   
C(condition):belief_human            0.157969    1.0   0.461842  4.976521e-01   
C(model):C(condition):belief_human   0.400214    3.0   0.390026  7.603254e-01   
feel_human                          11.632657    1.0  34.009579  2.551260e-08   
C(model):feel_human                  0.881770    3.0   0.859323  4.633788e-01   
C(condition):feel_human              0.874266    1.0   2.556031  1.116577e-01   
C(model):C(condition):

Shapiro-Wilk: 0.954, p = 1.16e-06***
Kolmogorov-Smirnov: 0.075, p = 0.15
[3-way model]
                              sum_sq     df          F        PR(>F)  \
C(condition)                1.752275    2.0   2.327432  9.996538e-02   
belief_human                0.331750    1.0   0.881285  3.488890e-01   
C(condition):belief_human   1.205724    2.0   1.601484  2.039596e-01   
feel_human                 17.896654    1.0  47.541905  5.732829e-11   
C(condition):feel_human     1.437638    2.0   1.909520  1.506212e-01   
Residual                   82.063823  218.0        NaN           NaN   

                                     p  partial_n2  
C(condition)                      0.10    0.020906  
belief_human                      0.35    0.004026  
C(condition):belief_human         0.20    0.014480  
feel_human                 5.73e-11***    0.179037  
C(condition):feel_human           0.15    0.017217  
Residual                           nan         NaN  
 
                             Contra

Shapiro-Wilk: 0.918, p = 3.76e-09***
Kolmogorov-Smirnov: 0.098, p = 0.040*
[2 x 4 model]
                                       sum_sq     df          F        PR(>F)  \
C(model)                             2.126131    3.0   1.796783  1.494996e-01   
C(condition)                         2.710675    1.0   6.872333  9.516056e-03   
C(model):C(condition)                1.225216    3.0   1.035424  3.782561e-01   
belief_human                         0.053137    1.0   0.134718  7.140285e-01   
C(model):belief_human                0.413045    3.0   0.349062  7.898711e-01   
C(condition):belief_human            0.286970    1.0   0.727550  3.948313e-01   
C(model):C(condition):belief_human   0.522055    3.0   0.441186  7.238333e-01   
feel_human                          12.500638    1.0  31.692679  6.969836e-08   
C(model):feel_human                  1.978089    3.0   1.671673  1.748099e-01   
C(condition):feel_human              1.184356    1.0   3.002681  8.486859e-02   
C(model):C(condition

Shapiro-Wilk: 0.910, p = 1.83e-10***
Kolmogorov-Smirnov: 0.105, p = 0.012*
[3-way model]
                              sum_sq     df          F        PR(>F)  \
C(condition)                2.266658    2.0   2.728054  6.757791e-02   
belief_human                0.001388    1.0   0.003340  9.539669e-01   
C(condition):belief_human   1.095610    2.0   1.318629  2.696267e-01   
feel_human                 18.979085    1.0  45.684843  1.251365e-10   
C(condition):feel_human     1.074492    2.0   1.293212  2.764843e-01   
Residual                   90.564840  218.0        NaN           NaN   

                                     p  partial_n2  
C(condition)                      0.07    0.024417  
belief_human                      0.95    0.000015  
C(condition):belief_human         0.27    0.011953  
feel_human                 1.25e-10***    0.173255  
C(condition):feel_human           0.28    0.011725  
Residual                           nan         NaN  
 
                             Cont

Shapiro-Wilk: 0.941, p = 2.53e-07***
Kolmogorov-Smirnov: 0.084, p = 0.11
[2 x 4 model]
                                       sum_sq     df          F        PR(>F)  \
C(model)                             1.855314    3.0   1.644088  1.809154e-01   
C(condition)                         3.800214    1.0  10.102688  1.747677e-03   
C(model):C(condition)                4.937816    3.0   4.375649  5.336246e-03   
belief_human                         0.103013    1.0   0.273855  6.014122e-01   
C(model):belief_human                1.318585    3.0   1.168465  3.232474e-01   
C(condition):belief_human            0.040934    1.0   0.108821  7.418810e-01   
C(model):C(condition):belief_human   0.401902    3.0   0.356145  7.847497e-01   
feel_human                          14.979515    1.0  39.822322  2.165551e-09   
C(model):feel_human                  2.820768    3.0   2.499625  6.113054e-02   
C(condition):feel_human              0.540323    1.0   1.436422  2.323214e-01   
C(model):C(condition):

Shapiro-Wilk: 0.922, p = 1.54e-09***
Kolmogorov-Smirnov: 0.137, p = 3.55e-04***
[3-way model]
                              sum_sq     df          F        PR(>F)  \
C(condition)                3.177392    2.0   3.535184  3.083620e-02   
belief_human                0.067329    1.0   0.149822  6.990832e-01   
C(condition):belief_human   0.317003    2.0   0.352699  7.031891e-01   
feel_human                 20.988097    1.0  46.702939  8.150778e-11   
C(condition):feel_human     1.258438    2.0   1.400145  2.487695e-01   
Residual                   97.968246  218.0        NaN           NaN   

                                     p  partial_n2  
C(condition)                    0.031*    0.031414  
belief_human                      0.70    0.000687  
C(condition):belief_human         0.70    0.003225  
feel_human                 8.15e-11***    0.176435  
C(condition):feel_human           0.25    0.012682  
Residual                           nan         NaN  
 
                            

Shapiro-Wilk: 0.970, p = 2.67e-04***
Kolmogorov-Smirnov: 0.072, p = 0.24
[2 x 4 model]
                                        sum_sq     df          F    PR(>F)  \
C(model)                              3.718821    3.0   1.916191  0.128644   
C(condition)                          1.022140    1.0   1.580029  0.210413   
C(model):C(condition)                 4.183737    3.0   2.155747  0.094944   
belief_human                          3.783080    1.0   5.847904  0.016609   
C(model):belief_human                 3.016159    3.0   1.554132  0.202260   
C(condition):belief_human             0.206427    1.0   0.319095  0.572867   
C(model):C(condition):belief_human    0.726334    3.0   0.374257  0.771675   
feel_human                            7.986515    1.0  12.345597  0.000561   
C(model):feel_human                   0.374509    3.0   0.192973  0.901086   
C(condition):feel_human               0.963771    1.0   1.489803  0.223871   
C(model):C(condition):feel_human      0.229711    3.0  

Shapiro-Wilk: 0.981, p = 0.003**
Kolmogorov-Smirnov: 0.066, p = 0.26
[3-way model]
                               sum_sq     df          F    PR(>F)  \
C(condition)                 1.566953    2.0   1.216420  0.298292   
belief_human                 2.267145    1.0   3.519954  0.061970   
C(condition):belief_human    2.919364    2.0   2.266292  0.106134   
feel_human                  14.089176    1.0  21.874755  0.000005   
C(condition):feel_human      2.143629    2.0   1.664092  0.191759   
Residual                   140.410272  218.0        NaN       NaN   

                                     p  partial_n2  
C(condition)                      0.30    0.011037  
belief_human                      0.06    0.015890  
C(condition):belief_human         0.11    0.020368  
feel_human                 5.10e-06***    0.091192  
C(condition):feel_human           0.19    0.015037  
Residual                           nan         NaN  
 
                             Contrast  Adj. mean diff       

### Supplementary: Controlling for latency & duration（补充：控制延迟与会话时长）

In [10]:
df = (
    df_users
    .copy()
    .merge(
        df_pids[['pid','condition','model','mean_response_latency_seconds','session_duration_minutes']],
        on='pid',
        how='left'
        )
)

cols = [x for x in df.columns if 'waisr_' in x]
for col in cols:
    df[col] = df[col].str.lower().replace({
        'strong disagree': 1,
        'disagree': 2,
        'neutral': 3,
        'agree': 4,
        'strong agree': 5,   
    }).astype(int)

df['waisr_overall'] = df[cols].mean(axis=1)

# ------------------------------------------------------------------------------------------------------------------------------------
# 运行模型
# ------------------------------------------------------------------------------------------------------------------------------------

outcome = 'overall'

# --- 2x4 模型
model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

model_fit = smf.ols(
    f'waisr_{outcome} ~ C(model) * C(condition) * belief_human + C(model) * C(condition) * feel_human + session_duration_minutes + mean_response_latency_seconds',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[2 x 4 model]')
print(anova_table)

# --- 三因素模型
model_df = df.groupby(['pid','condition'])[[f'waisr_{outcome}','belief_human','feel_human','session_duration_minutes','mean_response_latency_seconds']].mean().reset_index()

model_fit = smf.ols(
    f'waisr_{outcome} ~ C(condition)*belief_human + C(condition)*feel_human + session_duration_minutes + mean_response_latency_seconds',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[3-way model]')
print(anova_table)

print(' ')
print(anova_pairwise_comparisons(model_fit))


Shapiro-Wilk: 0.954, p = 3.99e-06***
Kolmogorov-Smirnov: 0.081, p = 0.14
[2 x 4 model]
                                       sum_sq     df          F        PR(>F)  \
C(model)                             1.801074    3.0   1.779943  1.527370e-01   
C(condition)                         1.414552    1.0   4.193867  4.206462e-02   
C(model):C(condition)                2.806619    3.0   2.773690  4.296528e-02   
belief_human                         0.374394    1.0   1.110003  2.935338e-01   
C(model):belief_human                0.839496    3.0   0.829647  4.791996e-01   
C(condition):belief_human            0.085740    1.0   0.254202  6.147656e-01   
C(model):C(condition):belief_human   0.654810    3.0   0.647127  5.857557e-01   
feel_human                          11.226488    1.0  33.284317  3.542583e-08   
C(model):feel_human                  0.805742    3.0   0.796288  4.974692e-01   
C(condition):feel_human              1.277876    1.0   3.788649  5.320252e-02   
C(model):C(condition):

Shapiro-Wilk: 0.952, p = 7.42e-07***
Kolmogorov-Smirnov: 0.076, p = 0.14
[3-way model]
                                  sum_sq     df          F        PR(>F)  \
C(condition)                    2.164177    2.0   2.946714  5.462677e-02   
belief_human                    0.162171    1.0   0.441620  5.070505e-01   
C(condition):belief_human       0.722697    2.0   0.984015  3.754767e-01   
feel_human                     15.814410    1.0  43.065368  3.865156e-10   
C(condition):feel_human         1.573474    2.0   2.142420  1.198580e-01   
session_duration_minutes        1.776661    1.0   4.838153  2.889729e-02   
mean_response_latency_seconds   0.420056    1.0   1.143885  2.860267e-01   
Residual                       79.319248  216.0        NaN           NaN   

                                         p  partial_n2  
C(condition)                          0.05    0.026560  
belief_human                          0.51    0.002040  
C(condition):belief_human             0.38    0.009029  


## Perception of humanness（人性化感知）

In [11]:
df = df_users.copy().merge(df_pids[['pid','condition','model']],on='pid',how='left')
for col in ['feel_human','belief_human']:
    df[col] = (df[col] - 1) / 6

df['ai'] = (df['condition']!='human_therapist').replace({True: 'AI', False: 'Human'})

display(
    df.groupby('ai')[['belief_human','feel_human']]
      .agg(['mean','std'])
      .map(lambda x: f"{x*100:.1f}%" if pd.notnull(x) else "")
)

# ------------------------------------------------------------------------------------------------------------------------------------
# 运行统计检验
# ------------------------------------------------------------------------------------------------------------------------------------

for outcome in ['feel_human','belief_human']:

    print_title(outcome)

    vec_A = df.loc[df['condition']=='cognitive_layer',outcome]
    vec_B = df.loc[df['condition']=='standalone_llms',outcome]
    vec_C = df.loc[df['condition']=='human_therapist',outcome]

    for combo in [('cognitive_layer','standalone_llms'),('cognitive_layer','human_therapist'),('standalone_llms','human_therapist')]:
        vec_A = df.loc[df['condition']==combo[0],outcome]
        vec_B = df.loc[df['condition']==combo[1],outcome]
        stat = sp.mannwhitneyu(vec_A, vec_B)
        print(f'{combo[0]} vs {combo[1]}: U = {stat.statistic}, p(bonf) = {readable_pvalue(stat.pvalue*6)}')

# ------------------------------------------------------------------------------------------------------------------------------------
# 绘图
# ------------------------------------------------------------------------------------------------------------------------------------
 
melted = (
    df
    .melt(id_vars=['pid','condition'],value_vars=['feel_human','belief_human'],var_name='variable',value_name='rating')
)

fig = plot_dot_and_bar(
    melted,
    outcome_var='rating',
    group_var='variable',
    condition_col='condition',
    items=['feel_human','belief_human'],
    condition_order=['cognitive_layer','standalone_llms','human_therapist'],
    COLOURS=COLOURS,
    score_range=(0,1),
    show_barometer=True,
)
fig.update_xaxes(range=[-0.5,1.5])
fig.update_yaxes(range=[0,1.1])
fig.write_image('../results/humanness_fig.svg',width=400,height=380)

# summary = (
#     df
#     .melt(id_vars=['pid','condition'],value_vars=['feel_human','belief_human'],var_name='variable',value_name='rating')
#     .groupby(['condition','variable'])['rating'].agg(['mean','sem'])
#     .reset_index()
# )

# fig = px.bar(
#     summary,
#     x='variable',
#     y='mean',
#     error_y='sem',
#     color='condition',
#     title='Ratings',
#     barmode='group',
#     width=400,
#     height=400,
#     labels={'mean': 'rating'},
#     category_orders={
#         'condition': CONDITION_ORDER,
#         'variable': ['feel_human','belief_human']
#         },
#     color_discrete_map=COLOURS,
#     orientation='v',
#     template='simple_white',
# )
# fig.update_traces(width=0.2)
# fig.update_yaxes(range=[0,1],tickformat=".0%")
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.update_layout(font={'family': 'Arial'})
# fig.add_hline(y=.5,line_dash='dot',line_color='black',line_width=1,opacity=1,layer='above')
# fig.show()

# fig.write_image(f'../results/humanness_ratings.svg',width=400,height=400)


belief_human        feel_human       
              mean    std       mean    std
ai                                         
AI           15.3%  25.4%      45.9%  29.1%
Human        37.2%  31.0%      52.6%  32.9%

FEEL HUMAN
cognitive_layer vs standalone_llms: U = 5496.0, p(bonf) = > .999
cognitive_layer vs human_therapist: U = 1204.5, p(bonf) = > .999
standalone_llms vs human_therapist: U = 1094.0, p(bonf) = > .999
BELIEF HUMAN
cognitive_layer vs standalone_llms: U = 4938.0, p(bonf) = > .999
cognitive_layer vs human_therapist: U = 760.0, p(bonf) = 0.002**
standalone_llms vs human_therapist: U = 770.5, p(bonf) = 0.003**


Resorting to unclean kill browser.


### Supplementary: Controlling for latency & duration（补充：控制延迟与会话时长）

In [12]:
df = df_users.copy().merge(df_pids[['pid','condition','model','mean_response_latency_seconds','session_duration_minutes']],on='pid',how='left')
for col in ['feel_human','belief_human']:
    df[col] = (df[col] - 1) / 6

df['ai'] = (df['condition']!='human_therapist').replace({True: 'AI', False: 'Human'})

display(
    df.groupby('ai')[['belief_human','feel_human']]
      .agg(['mean','std'])
      .map(lambda x: f"{x*100:.1f}%" if pd.notnull(x) else "")
)

# ------------------------------------------------------------------------------------------------------------------------------------
# 运行模型
# ------------------------------------------------------------------------------------------------------------------------------------

for outcome in ['feel_human','belief_human']:

    print_title(outcome)

    # --- 2x4 模型
    model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

    model_fit = smf.ols(
        f'{outcome} ~ + C(model) * C(condition) + session_duration_minutes + mean_response_latency_seconds',
        data=model_df).fit()

    display_residuals(model_fit)
    anova_table = format_anova_table(model_fit)

    print('[2 x 4 model]')
    print(anova_table)

    # --- 三因素模型
    model_df = df.groupby(['pid','condition'])[[outcome,'session_duration_minutes','mean_response_latency_seconds']].mean().reset_index()

    model_fit = smf.ols(
        f'{outcome} ~ C(condition) + session_duration_minutes + mean_response_latency_seconds',
        data=model_df).fit()

    display_residuals(model_fit)
    anova_table = format_anova_table(model_fit)

    print('[3-way model]')
    print(anova_table)

    print(' ')
    print(anova_pairwise_comparisons(model_fit))

belief_human        feel_human       
              mean    std       mean    std
ai                                         
AI           15.3%  25.4%      45.9%  29.1%
Human        37.2%  31.0%      52.6%  32.9%

FEEL HUMAN


Shapiro-Wilk: 0.980, p = 0.007**
Kolmogorov-Smirnov: 0.059, p = 0.48
[2 x 4 model]
                                  sum_sq     df         F    PR(>F)       p  \
C(model)                        0.077038    3.0  0.317884  0.812438    0.81   
C(condition)                    0.010259    1.0  0.127002  0.721955    0.72   
C(model):C(condition)           0.485361    3.0  2.002774  0.115008    0.12   
session_duration_minutes        0.396174    1.0  4.904269  0.027973  0.028*   
mean_response_latency_seconds   0.001015    1.0  0.012561  0.910883    0.91   
Residual                       15.429275  191.0       NaN       NaN     nan   

                               partial_n2  
C(model)                         0.004968  
C(condition)                     0.000664  
C(model):C(condition)            0.030498  
session_duration_minutes         0.025034  
mean_response_latency_seconds    0.000066  
Residual                              NaN  


Shapiro-Wilk: 0.971, p = 1.28e-04***
Kolmogorov-Smirnov: 0.069, p = 0.22
[3-way model]
                                  sum_sq     df         F    PR(>F)       p  \
C(condition)                    0.171424    2.0  1.003611  0.368210    0.37   
session_duration_minutes        0.378253    1.0  4.429014  0.036458  0.036*   
mean_response_latency_seconds   0.144205    1.0  1.688514  0.195145    0.20   
Residual                       18.959582  222.0       NaN       NaN     nan   

                               partial_n2  
C(condition)                     0.008961  
session_duration_minutes         0.019560  
mean_response_latency_seconds    0.007549  
Residual                              NaN  
 
                             Contrast  Adj. mean diff                     SE  \
0  cognitive layer vs standalone LLMs        0.006111  [0.04563523902136056]   
1  cognitive layer vs human therapist       -0.222651   [0.1578511235487847]   
2  standalone LLMs vs human therapist       -0.228762  

Shapiro-Wilk: 0.804, p = 3.69e-15***
Kolmogorov-Smirnov: 0.224, p = 2.52e-09***
[2 x 4 model]
                                  sum_sq     df          F    PR(>F)  \
C(model)                        0.023589    3.0   0.124471  0.945555   
C(condition)                    0.069594    1.0   1.101683  0.295223   
C(model):C(condition)           0.020996    3.0   0.110792  0.953716   
session_duration_minutes        0.743946    1.0  11.776847  0.000735   
mean_response_latency_seconds   0.000908    1.0   0.014370  0.904709   
Residual                       12.065518  191.0        NaN       NaN   

                                         p  partial_n2  
C(model)                              0.95    0.001951  
C(condition)                          0.30    0.005735  
C(model):C(condition)                 0.95    0.001737  
session_duration_minutes       7.35e-04***    0.058078  
mean_response_latency_seconds         0.90    0.000075  
Residual                               nan         NaN  


Shapiro-Wilk: 0.850, p = 4.53e-14***
Kolmogorov-Smirnov: 0.202, p = 1.24e-08***
[3-way model]
                                  sum_sq     df          F    PR(>F)  \
C(condition)                    0.313767    2.0   2.391483  0.093847   
session_duration_minutes        0.746746    1.0  11.383174  0.000874   
mean_response_latency_seconds   0.028709    1.0   0.437630  0.508954   
Residual                       14.563381  222.0        NaN       NaN   

                                         p  partial_n2  
C(condition)                          0.09    0.021091  
session_duration_minutes       8.74e-04***    0.048775  
mean_response_latency_seconds         0.51    0.001967  
Residual                               nan         NaN  
 
                             Contrast  Adj. mean diff                      SE  \
0  cognitive layer vs standalone LLMs        0.065907  [0.039996045968473365]   
1  cognitive layer vs human therapist        0.198770   [0.13834529913773932]   
2  standalone L